# DATA 266 Lab 1 — Part 2: Yelp Polarity Sentiment Classification

**VS Code / local / RunPod-ready version**

This notebook trains **three models from scratch**:
1. Baseline: Mean Embedding + MLP
2. Experimental 1: TextCNN
3. Experimental 2: BiLSTM

No pretrained language model and no pretrained text embedding is used.

The notebook automatically uses the current Git repository as `REPO_ROOT` when run from VS Code.
You can optionally override it with the `DATA266_REPO_ROOT` environment variable.


In [ ]:
# Install dependencies into the SAME Python environment used by this VS Code notebook.
import sys
import subprocess

packages = [
    "datasets",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "scipy",
    "nltk",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", *packages
])

print("Python:", sys.executable)


In [2]:
from pathlib import Path
import os, sys, re, json, math, time, random, logging, platform
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    roc_auc_score, average_precision_score, matthews_corrcoef, brier_score_loss,
    f1_score
)
from scipy.stats import binomtest

SEED = 266
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer


Device: cuda
GPU: Tesla T4


In [ ]:
# ===== VS CODE / LOCAL / RUNPOD REPOSITORY SETUP =====
from pathlib import Path
import os
import subprocess

# Optional override. Example:
# export DATA266_REPO_ROOT="/root/workplace/Lab1_DATA266"
repo_env = os.environ.get("DATA266_REPO_ROOT", "").strip()

if repo_env:
    REPO_ROOT = Path(repo_env).expanduser().resolve()
else:
    # In VS Code, use the root of the currently opened Git repository.
    try:
        git_root = subprocess.check_output(
            ["git", "rev-parse", "--show-toplevel"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        REPO_ROOT = Path(git_root).resolve()
    except Exception:
        # Fallback to the current working directory if this is not a Git repository.
        REPO_ROOT = Path.cwd().resolve()

MEMBER_NAME = os.environ.get("DATA266_MEMBER_NAME", "Kavya").strip()
RUN_MODE = os.environ.get("DATA266_RUN_MODE", "full").strip().lower()

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("DATA266_RUN_MODE must be 'smoke' or 'full'.")

print("Repository root:", REPO_ROOT)
print("Member name:", MEMBER_NAME)
print("Run mode:", RUN_MODE)

TASK_ROOT = REPO_ROOT / "task2_sentiment"
MEMBER_DIR = TASK_ROOT / MEMBER_NAME
SRC_DIR = MEMBER_DIR / "src"
DATA_PROCESSED_DIR = MEMBER_DIR / "data_processed"
CKPT_DIR = MEMBER_DIR / "checkpoints"
OUTPUT_DIR = MEMBER_DIR / "outputs"
MANIFEST_DIR = REPO_ROOT / "reproducibility" / "manifests"
RAW_LOG_DIR = REPO_ROOT / "reproducibility" / "raw_logs"

RUN_CKPT_DIR = CKPT_DIR / RUN_MODE
RUN_OUTPUT_DIR = OUTPUT_DIR if RUN_MODE == "full" else OUTPUT_DIR / "smoke"
ARTIFACT_BASE = MEMBER_DIR if RUN_MODE == "full" else RUN_OUTPUT_DIR

for d in [
    MEMBER_DIR,
    SRC_DIR,
    DATA_PROCESSED_DIR,
    CKPT_DIR,
    RUN_CKPT_DIR,
    OUTPUT_DIR,
    RUN_OUTPUT_DIR,
    MANIFEST_DIR,
    RAW_LOG_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_DIR = RUN_CKPT_DIR

MAX_TRAIN_ROWS = 200_000 if RUN_MODE == "full" else 10_000
MAX_TEST_ROWS = 38_000 if RUN_MODE == "full" else 2_000
MAX_VOCAB = 50_000
MAX_LEN = 256
MIN_FREQ = 2
BATCH_SIZE = 256
EPOCHS = 5 if RUN_MODE == "full" else 1
EMBED_DIM = 128
LEARNING_RATE = 2e-3
NUM_WORKERS = 2
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
PREPROCESS_VERSION = "escaped_sequences_v1"

SENTIMENT_KEEP_WORDS = {
    "no", "not", "nor", "never",
    "cannot", "nothing", "none", "nobody", "neither", "without",
    "but", "very", "too", "few", "less", "against",
}

PREPROCESS_MODE = os.environ.get(
    "DATA266_PREPROCESS_MODE",
    "stop_stem"
).strip().lower()

if PREPROCESS_MODE not in {"basic", "stop_stem"}:
    raise ValueError(
        "DATA266_PREPROCESS_MODE must be 'basic' or 'stop_stem'."
    )

CONFIG = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "max_train_rows": MAX_TRAIN_ROWS,
    "max_test_rows": MAX_TEST_ROWS,
    "max_vocab": MAX_VOCAB,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "embedding_dim": EMBED_DIM,
    "learning_rate": LEARNING_RATE,
    "preprocess_mode": PREPROCESS_MODE,
    "sentiment_keep_words": sorted(SENTIMENT_KEEP_WORDS),
    "device": str(DEVICE),
    "hardware": GPU_NAME,
    "preprocess_version": PREPROCESS_VERSION,
}

CONFIG_PATH = (
    MEMBER_DIR / "config.json"
    if RUN_MODE == "full"
    else RUN_OUTPUT_DIR / "config_smoke.json"
)

CONFIG_PATH.write_text(
    json.dumps(CONFIG, indent=2),
    encoding="utf-8"
)

print("Artifacts directory:", ARTIFACT_BASE)
print("Checkpoint directory:", RUN_CKPT_DIR)
print("Hardware:", GPU_NAME)

def assert_config_compatible(saved_config, current_config, label="checkpoint"):
    """Refuse to resume when training-relevant settings differ."""
    ignored = {"device"}
    keys = sorted((set(saved_config) | set(current_config)) - ignored)
    mismatches = []

    for key in keys:
        if saved_config.get(key) != current_config.get(key):
            mismatches.append(
                f"{key}: saved={saved_config.get(key)!r}, "
                f"current={current_config.get(key)!r}"
            )

    if mismatches:
        details = "\n  - " + "\n  - ".join(mismatches)
        raise RuntimeError(
            f"Refusing to reuse {label}: its configuration does not match "
            f"this run.{details}\n"
            "Use the correct RUN_MODE/config, or intentionally remove that "
            "run's checkpoint directory."
        )


In [4]:
LOG_PATH = RAW_LOG_DIR / f"{MEMBER_NAME}_task2_{RUN_MODE}.log"
logger = logging.getLogger(f"part2_{RUN_MODE}")
logger.setLevel(logging.INFO)
logger.handlers.clear()
fh = logging.FileHandler(LOG_PATH, mode="a")
sh = logging.StreamHandler(sys.stdout)
fmt = logging.Formatter("%(asctime)s | %(message)s")
fh.setFormatter(fmt); sh.setFormatter(fmt)
logger.addHandler(fh); logger.addHandler(sh)
logger.info("Part 2 run/session started")
logger.info(f"Run mode={RUN_MODE} | preprocessing={PREPROCESS_MODE}")
logger.info(f"Python={platform.python_version()} | PyTorch={torch.__version__} | Device={DEVICE}")
if torch.cuda.is_available():
    logger.info(f"GPU={torch.cuda.get_device_name(0)}")

2026-10-02 18:53:59,771 | Part 2 run/session started


INFO:part2_full:Part 2 run/session started


2026-10-02 18:53:59,775 | Run mode=full | preprocessing=stop_stem


INFO:part2_full:Run mode=full | preprocessing=stop_stem


2026-10-02 18:53:59,780 | Python=3.13.15 | PyTorch=2.11.0+cu130 | Device=cuda


INFO:part2_full:Python=3.13.15 | PyTorch=2.11.0+cu130 | Device=cuda


2026-10-02 18:53:59,789 | GPU=Tesla T4


INFO:part2_full:GPU=Tesla T4


## 1. Load Yelp Polarity

In [5]:
ds = load_dataset("fancyzhx/yelp_polarity")

train_raw = ds["train"].shuffle(seed=SEED)
test_raw = ds["test"].shuffle(seed=SEED)

if MAX_TRAIN_ROWS is not None:
    train_raw = train_raw.select(range(min(MAX_TRAIN_ROWS, len(train_raw))))
if MAX_TEST_ROWS is not None:
    test_raw = test_raw.select(range(min(MAX_TEST_ROWS, len(test_raw))))

train_df = train_raw.to_pandas()[["text", "label"]].copy()
test_df = test_raw.to_pandas()[["text", "label"]].copy()

print(train_df.shape, test_df.shape)
train_df.head()

(200000, 2) (38000, 2)


,text,label
0,"Sorry, but I have to do this.\n\nI have had go...",0
1,My cousin found out about this place on Guy Fi...,1
2,They're slipping something fierce. One of the...,0
3,Imagine Montreal: green areas and fountain pon...,1
4,"I thought Bite was great! There were some \""""c...",1


## 2. Data analysis and preprocessing

In [6]:
def basic_stats(df, name):
    lengths = df["text"].fillna("").str.split().str.len()
    print(f"{name} rows:", len(df))
    print("Missing text:", df["text"].isna().sum())
    print("Class distribution:")
    print(df["label"].value_counts(normalize=True).sort_index())
    print("Review length percentiles:")
    print(lengths.quantile([0.50, 0.90, 0.95, 0.99]))

basic_stats(train_df, "Train")
basic_stats(test_df, "Test")

Train rows: 200000
Missing text: 0
Class distribution:
label
0    0.500615
1    0.499385
Name: proportion, dtype: float64
Review length percentiles:
0.50     97.0
0.90    281.0
0.95    372.0
0.99    612.0
Name: text, dtype: float64
Test rows: 38000
Missing text: 0
Class distribution:
label
0    0.5
1    0.5
Name: proportion, dtype: float64
Review length percentiles:
0.50     97.00
0.90    281.00
0.95    367.00
0.99    603.01
Name: text, dtype: float64


In [7]:
# Two explicit preprocessing pipelines are implemented so their effect can be compared.
TOKEN_RE = re.compile(r"[a-z0-9']+")
STOP_WORDS = set(ENGLISH_STOP_WORDS)
STEMMER = PorterStemmer()

SENTIMENT_KEEP_WORDS = {
    "no", "not", "nor", "never",
    "cannot", "nothing", "none", "nobody", "neither", "without",
    "but", "very", "too", "few", "less", "against",
}

ESCAPED_UNICODE_RE = re.compile(r"\\u[0-9a-fA-F]{4}")
ESCAPED_HEX_RE = re.compile(r"\\x[0-9a-fA-F]{2}")

def clean_text(text):
    text = "" if text is None else str(text)
    text = re.sub(r"\\[nrt]", " ", text)
    text = ESCAPED_UNICODE_RE.sub(" ", text)
    text = ESCAPED_HEX_RE.sub(" ", text)
    return text

def preprocess(text, mode=PREPROCESS_MODE):
    text = clean_text(text).lower()
    tokens = TOKEN_RE.findall(text)

    if mode == "basic":
        return tokens

    if mode == "stop_stem":
        tokens = [
            t for t in tokens
            if t not in STOP_WORDS or t in SENTIMENT_KEEP_WORDS
        ]
        return [STEMMER.stem(t) for t in tokens]

    raise ValueError(f"Unknown preprocessing mode: {mode}")

example = r"Nothing special,\n but very good fries \u3001 and not too salty."
print("Cleaned example:", clean_text(example))
print("Preserved sentiment words:", sorted(SENTIMENT_KEEP_WORDS))
for mode in ["basic", "stop_stem"]:
    print(mode, "->", preprocess(example, mode))

(ARTIFACT_BASE / "sentiment_stopwords_preserved.json").write_text(
    json.dumps(
        {"preserved_from_stopword_removal": sorted(SENTIMENT_KEEP_WORDS)},
        indent=2
    ),
    encoding="utf-8"
)

(DATA_PROCESSED_DIR / "preprocessing_summary.json").write_text(
    json.dumps(
        {
            "version": PREPROCESS_VERSION,
            "escaped_newline_tab_cleanup": True,
            "escaped_unicode_cleanup": True,
            "escaped_hex_cleanup": True,
            "final_mode": PREPROCESS_MODE,
        },
        indent=2
    ),
    encoding="utf-8"
)

Cleaned example: Nothing special,  but very good fries   and not too salty.
Preserved sentiment words: ['against', 'but', 'cannot', 'few', 'less', 'neither', 'never', 'no', 'nobody', 'none', 'nor', 'not', 'nothing', 'too', 'very', 'without']
basic -> ['nothing', 'special', 'but', 'very', 'good', 'fries', 'and', 'not', 'too', 'salty']
stop_stem -> ['noth', 'special', 'but', 'veri', 'good', 'fri', 'not', 'too', 'salti']


173

In [8]:
def build_vocab(texts, mode, max_vocab=MAX_VOCAB, min_freq=MIN_FREQ):
    counter = Counter()
    for text in texts:
        counter.update(preprocess(text, mode))
    specials = ["<pad>", "<unk>"]
    vocab_words = [w for w, c in counter.most_common(max_vocab - len(specials)) if c >= min_freq]
    itos_local = specials + vocab_words
    stoi_local = {w: i for i, w in enumerate(itos_local)}
    return stoi_local, itos_local, counter

stoi, itos, counter = build_vocab(train_df["text"], PREPROCESS_MODE)
PAD_ID = 0
UNK_ID = 1
VOCAB_SIZE = len(itos)

print("Final preprocessing mode:", PREPROCESS_MODE)
print("Vocabulary size:", VOCAB_SIZE)
print("Most common:", counter.most_common(20))
print("Count of junk token 'n':", counter.get("n", 0))

pd.DataFrame(counter.most_common(100), columns=["token", "count"]).to_csv(
    DATA_PROCESSED_DIR / "vocab_top100.csv", index=False
)
(DATA_PROCESSED_DIR / "vocab.json").write_text(
    json.dumps(itos, ensure_ascii=False),
    encoding="utf-8"
)

Final preprocessing mode: stop_stem
Vocabulary size: 50000
Most common: [('but', 206849), ('not', 183089), ('place', 130031), ('food', 121404), ('good', 110200), ('like', 103461), ('time', 100050), ('just', 92987), ('order', 85868), ('veri', 85202), ('servic', 80280), ('great', 76345), ('no', 69323), ("it'", 61458), ('realli', 61050), ('tri', 57678), ("don't", 49272), ('look', 48924), ('want', 48781), ('got', 48143)]
Count of junk token 'n': 3339


512992

In [9]:
class YelpDataset(Dataset):
    def __init__(self, df, stoi_map=None, mode=None, max_len=MAX_LEN):
        self.texts = df["text"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.stoi = stoi if stoi_map is None else stoi_map
        self.mode = PREPROCESS_MODE if mode is None else mode
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        tokens = preprocess(self.texts[idx], self.mode)[:self.max_len]
        ids = [self.stoi.get(tok, UNK_ID) for tok in tokens]
        if len(ids) < self.max_len:
            ids += [PAD_ID] * (self.max_len - len(ids))
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
            self.texts[idx],
        )

train_ds = YelpDataset(train_df)
test_ds = YelpDataset(test_df)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
x, y, txt = next(iter(train_loader))
print(x.shape, y.shape)

torch.Size([256, 256]) torch.Size([256])


## 3. Three models

In [10]:
class MeanEmbeddingMLP(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.net = nn.Sequential(
            nn.Linear(emb_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )

    def forward(self, x):
        emb = self.embedding(x)                          # [B,T,E]
        mask = (x != PAD_ID).float().unsqueeze(-1)     # [B,T,1]
        pooled = (emb * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        return self.net(pooled)


class TextCNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, channels=128, kernels=(3,4,5), dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.convs = nn.ModuleList([
            nn.Conv1d(emb_dim, channels, kernel_size=k)
            for k in kernels
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(channels * len(kernels), 2)

    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)  # [B,E,T]
        feats = [F.relu(conv(emb)).amax(dim=-1) for conv in self.convs]
        z = torch.cat(feats, dim=1)
        return self.fc(self.dropout(z))


class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden=128, layers=1, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(
            emb_dim, hidden, num_layers=layers, batch_first=True,
            bidirectional=True, dropout=dropout if layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, 2)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)

        mask = (x != PAD_ID).float().unsqueeze(-1)
        pooled = (out * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        return self.fc(self.dropout(pooled))


MODEL_FACTORIES = {
    "baseline_mean_mlp": lambda: MeanEmbeddingMLP(VOCAB_SIZE, EMBED_DIM, hidden=128, dropout=0.30),
    "experimental_textcnn": lambda: TextCNN(VOCAB_SIZE, EMBED_DIM, channels=128, kernels=(3,4,5), dropout=0.40),
    "experimental_bilstm": lambda: BiLSTMClassifier(VOCAB_SIZE, EMBED_DIM, hidden=128, layers=1, dropout=0.35),
}

for name, factory in MODEL_FACTORIES.items():
    m = factory()
    print(name, f"{sum(p.numel() for p in m.parameters()):,} parameters")

baseline_mean_mlp 6,416,770 parameters
experimental_textcnn 6,597,762 parameters
experimental_bilstm 6,664,706 parameters


In [11]:
# Lightweight evidence for the preprocessing choice: train the same baseline architecture
# for one epoch under both preprocessing modes on the same deterministic subset.
def preprocessing_ablation(train_frame, max_rows=20_000 if RUN_MODE == "full" else 4_000):
    work = train_frame.iloc[:min(max_rows, len(train_frame))].copy().reset_index(drop=True)
    split = int(0.8 * len(work))
    ab_train = work.iloc[:split].copy()
    ab_val = work.iloc[split:].copy()
    rows = []

    for mode in ["basic", "stop_stem"]:
        a_stoi, a_itos, _ = build_vocab(ab_train["text"], mode, max_vocab=min(MAX_VOCAB, 20_000))
        a_train_ds = YelpDataset(ab_train, stoi_map=a_stoi, mode=mode, max_len=min(MAX_LEN, 192))
        a_val_ds = YelpDataset(ab_val, stoi_map=a_stoi, mode=mode, max_len=min(MAX_LEN, 192))
        a_train_loader = DataLoader(a_train_ds, batch_size=BATCH_SIZE, shuffle=True)
        a_val_loader = DataLoader(a_val_ds, batch_size=BATCH_SIZE, shuffle=False)

        m = MeanEmbeddingMLP(len(a_itos), EMBED_DIM, hidden=128, dropout=0.30).to(DEVICE)
        opt = torch.optim.AdamW(m.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        m.train()
        for xb, yb, _ in a_train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = crit(m(xb), yb)
            loss.backward()
            opt.step()

        m.eval()
        true, pred = [], []
        with torch.no_grad():
            for xb, yb, _ in a_val_loader:
                logits = m(xb.to(DEVICE))
                true.extend(yb.numpy().tolist())
                pred.extend(logits.argmax(1).cpu().numpy().tolist())
        rows.append({
            "preprocess_mode": mode,
            "ablation_train_rows": len(ab_train),
            "ablation_val_rows": len(ab_val),
            "vocab_size": len(a_itos),
            "accuracy": accuracy_score(true, pred),
            "macro_f1": f1_score(true, pred, average="macro"),
        })
        del m
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)

preprocessing_ablation_df = preprocessing_ablation(train_df)
preprocessing_ablation_df.to_csv(ARTIFACT_BASE / "preprocessing_ablation.csv", index=False)
print("Configured final preprocessing mode:", PREPROCESS_MODE)
preprocessing_ablation_df

Configured final preprocessing mode: stop_stem


,preprocess_mode,ablation_train_rows,ablation_val_rows,vocab_size,accuracy,macro_f1
0,basic,16000,4000,20000,0.8245,0.824499
1,stop_stem,16000,4000,15716,0.8455,0.845282


## 4. Train and evaluate

In [12]:
def expected_calibration_error(y_true, probs, n_bins=15):
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)
    conf = np.maximum(probs, 1 - probs)
    pred = (probs >= 0.5).astype(int)
    correct = (pred == y_true).astype(float)

    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)

def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=500, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        vals.append(metric_fn(np.asarray(y_true)[idx], np.asarray(y_pred)[idx]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def mcnemar_exact(y_true, pred_a, pred_b):
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)
    a_correct = pred_a == y_true
    b_correct = pred_b == y_true
    b = np.sum(a_correct & ~b_correct)
    c = np.sum(~a_correct & b_correct)
    if b + c == 0:
        return {"b": int(b), "c": int(c), "p_value": 1.0}
    p = binomtest(k=min(b, c), n=b+c, p=0.5, alternative="two-sided").pvalue
    return {"b": int(b), "c": int(c), "p_value": float(p)}

@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    ys, probs, preds, texts = [], [], [], []
    for x, y, batch_texts in loader:
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        p = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        pred = (p >= 0.5).astype(int)

        ys.extend(y.numpy().tolist())
        probs.extend(p.tolist())
        preds.extend(pred.tolist())
        texts.extend(list(batch_texts))
    return np.array(ys), np.array(probs), np.array(preds), texts

def train_one(name, model):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    history = []
    total_examples = 0
    start_all = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        n = 0
        correct = 0
        t0 = time.perf_counter()

        for x, y, _ in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            bs = y.size(0)
            loss_sum += loss.item() * bs
            correct += (logits.argmax(1) == y).sum().item()
            n += bs
            total_examples += bs

        elapsed = time.perf_counter() - t0
        row = {
            "epoch": epoch,
            "train_loss": loss_sum / n,
            "train_accuracy": correct / n,
            "examples_per_sec": n / max(elapsed, 1e-9),
        }
        history.append(row)
        logger.info(
            f"{name} | epoch={epoch}/{EPOCHS} | loss={row['train_loss']:.4f} "
            f"| acc={row['train_accuracy']:.4f} | ex/s={row['examples_per_sec']:.1f}"
        )

    train_seconds = time.perf_counter() - start_all
    peak_mem = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "model_name": name,
            "config": CONFIG,
            "vocab": itos,
            "training_time_seconds": train_seconds,
            "peak_memory_mb": peak_mem,
        },
        MODEL_DIR / f"{name}.pt"
    )

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(RUN_OUTPUT_DIR / f"training_history_{name}.csv", index=False)

    y_true, p_pos, y_pred, texts = predict_model(model, test_loader)
    return model, y_true, p_pos, y_pred, texts, train_seconds, peak_mem, hist_df

runs = {}
for name, factory in MODEL_FACTORIES.items():
    print("\nMODEL:", name)
    ckpt_path = MODEL_DIR / f"{name}.pt"
    if ckpt_path.exists():
        print("Checkpoint found; validating configuration:", ckpt_path)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        try:
            assert_config_compatible(ckpt.get("config", {}), CONFIG, label=str(ckpt_path))
            model = factory().to(DEVICE)
            model.load_state_dict(ckpt["model_state_dict"])
            hist_path = RUN_OUTPUT_DIR / f"training_history_{name}.csv"
            hist_df = pd.read_csv(hist_path) if hist_path.exists() else pd.DataFrame()
            train_seconds = float(ckpt.get("training_time_seconds", 0.0))
            peak_mem = float(ckpt.get("peak_memory_mb", 0.0))
            y_true, p_pos, y_pred, texts = predict_model(model, test_loader)
            runs[name] = (model, y_true, p_pos, y_pred, texts, train_seconds, peak_mem, hist_df)
            continue
        except RuntimeError as e:
            print(e)
            print("Configuration changed. Retraining this model.")

    print("Training:", name)
    runs[name] = train_one(name, factory())



MODEL: baseline_mean_mlp
Training: baseline_mean_mlp
2026-10-02 19:00:54,862 | baseline_mean_mlp | epoch=1/5 | loss=0.2598 | acc=0.8926 | ex/s=1117.7


INFO:part2_full:baseline_mean_mlp | epoch=1/5 | loss=0.2598 | acc=0.8926 | ex/s=1117.7


2026-10-02 19:03:53,130 | baseline_mean_mlp | epoch=2/5 | loss=0.1805 | acc=0.9312 | ex/s=1121.9


INFO:part2_full:baseline_mean_mlp | epoch=2/5 | loss=0.1805 | acc=0.9312 | ex/s=1121.9


2026-10-02 19:06:51,060 | baseline_mean_mlp | epoch=3/5 | loss=0.1594 | acc=0.9399 | ex/s=1124.1


...

INFO:part2_full:experimental_bilstm | epoch=3/5 | loss=0.1050 | acc=0.9608 | ex/s=1050.3


2026-10-02 19:42:17,586 | experimental_bilstm | epoch=4/5 | loss=0.0737 | acc=0.9737 | ex/s=1064.5


INFO:part2_full:experimental_bilstm | epoch=4/5 | loss=0.0737 | acc=0.9737 | ex/s=1064.5


2026-10-02 19:45:27,251 | experimental_bilstm | epoch=5/5 | loss=0.0501 | acc=0.9827 | ex/s=1054.5


INFO:part2_full:experimental_bilstm | epoch=5/5 | loss=0.0501 | acc=0.9827 | ex/s=1054.5


## 5. Required metrics

In [4]:
def slice_name(text):
    toks = preprocess(text)
    n = len(toks)
    if n < 50:
        return "short"
    if n < 150:
        return "medium"
    return "long"

def compute_metrics(name, y_true, probs, preds, texts, train_seconds, peak_mem, model):
    out = {"model": name}
    out["accuracy"] = accuracy_score(y_true, preds)

    for avg in ["macro", "micro", "weighted"]:
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, preds, average=avg, zero_division=0
        )
        out[f"precision_{avg}"] = p
        out[f"recall_{avg}"] = r
        out[f"f1_{avg}"] = f1

    out["roc_auc"] = roc_auc_score(y_true, probs)
    out["pr_auc"] = average_precision_score(y_true, probs)
    out["mcc"] = matthews_corrcoef(y_true, preds)
    out["brier_score"] = brier_score_loss(y_true, probs)
    out["expected_calibration_error"] = expected_calibration_error(y_true, probs)
    out["parameter_count"] = sum(p.numel() for p in model.parameters())
    out["training_time_seconds"] = train_seconds
    out["peak_memory_mb"] = peak_mem
    out["hardware"] = GPU_NAME

    acc_ci = bootstrap_ci(y_true, preds, accuracy_score)
    f1_ci = bootstrap_ci(y_true, preds, lambda a,b: f1_score(a,b,average="macro"))
    mcc_ci = bootstrap_ci(y_true, preds, matthews_corrcoef)

    out["accuracy_ci95_low"], out["accuracy_ci95_high"] = acc_ci
    out["macro_f1_ci95_low"], out["macro_f1_ci95_high"] = f1_ci
    out["mcc_ci95_low"], out["mcc_ci95_high"] = mcc_ci

    slices = np.array([slice_name(t) for t in texts])
    for s in ["short", "medium", "long"]:
        mask = slices == s
        if mask.any():
            out[f"slice_{s}_macro_f1"] = f1_score(y_true[mask], preds[mask], average="macro")
            out[f"slice_{s}_error_rate"] = 1 - accuracy_score(y_true[mask], preds[mask])

    return out

all_metrics = []
for name, (model, y_true, probs, preds, texts, train_seconds, peak_mem, hist_df) in runs.items():
    all_metrics.append(
        compute_metrics(name, y_true, probs, preds, texts, train_seconds, peak_mem, model)
    )

metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(ARTIFACT_BASE / "metrics_report.csv", index=False)
metrics_df

NameError: name 'runs' is not defined

## 6. Confusion matrices and ROC curves

In [3]:
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

for name, run in runs.items():
    model, y_true, probs, preds, texts, *_ = run

    ConfusionMatrixDisplay.from_predictions(y_true, preds, cmap="Blues")
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.savefig(RUN_OUTPUT_DIR / f"confusion_{name}.png", dpi=160)
    plt.show()

    RocCurveDisplay.from_predictions(y_true, probs)
    plt.title(f"ROC Curve — {name}")
    plt.tight_layout()
    plt.savefig(RUN_OUTPUT_DIR / f"roc_{name}.png", dpi=160)
    plt.show()

NameError: name 'runs' is not defined

## 7. Paired McNemar tests

In [2]:
baseline_pred = runs["baseline_mean_mlp"][3]
y_true = runs["baseline_mean_mlp"][1]

mcnemar_rows = []
for exp_name in ["experimental_textcnn", "experimental_bilstm"]:
    result = mcnemar_exact(y_true, baseline_pred, runs[exp_name][3])
    result["comparison"] = f"baseline_mean_mlp vs {exp_name}"
    mcnemar_rows.append(result)

mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv(ARTIFACT_BASE / "mcnemar_tests.csv", index=False)
mcnemar_df

NameError: name 'runs' is not defined

## 8. Select 20 errors for manual review

Required groups:
- 5 confident false positives
- 5 confident false negatives
- 5 near-threshold errors
- 5 slice-specific failures

The notebook automatically selects candidates. **You must manually inspect them and fill in**
`manual_error_type` and `testable_fix`.

In [1]:
# Use the best macro-F1 model for the 20-case error review.
best_name = metrics_df.sort_values("f1_macro", ascending=False).iloc[0]["model"]
model, y_true, probs, preds, texts, *_ = runs[best_name]

err = pd.DataFrame({
    "text": texts,
    "y_true": y_true,
    "prob_positive": probs,
    "y_pred": preds,
})

err["is_error"] = err["y_true"] != err["y_pred"]
err["slice"] = err["text"].map(slice_name)
err["confidence"] = np.where(
    err["y_pred"] == 1,
    err["prob_positive"],
    1 - err["prob_positive"]
)
err["distance_from_threshold"] = np.abs(err["prob_positive"] - 0.5)

fp = err[(err.y_true == 0) & (err.y_pred == 1)].nlargest(5, "confidence").copy()
fp["review_group"] = "confident_false_positive"

fn = err[(err.y_true == 1) & (err.y_pred == 0)].nlargest(5, "confidence").copy()
fn["review_group"] = "confident_false_negative"

near = err[err.is_error].nsmallest(5, "distance_from_threshold").copy()
near["review_group"] = "near_threshold_error"

slice_fail = err[
    err.is_error & (err["slice"] == "long")
].nlargest(5, "confidence").copy()

if len(slice_fail) < 5:
    extra = (
        err[err.is_error]
        .drop(slice_fail.index)
        .nlargest(5 - len(slice_fail), "confidence")
    )
    slice_fail = pd.concat([slice_fail, extra])

slice_fail["review_group"] = "slice_specific_failure"

review = pd.concat(
    [fp, fn, near, slice_fail],
    ignore_index=True
)


# -------- AUTO ERROR ANALYSIS --------

def auto_error_analysis(row):
    text = str(row["text"]).lower()
    group = row["review_group"]

    contrast_words = ["but", "although", "however", "though", "yet", "except"]
    sarcasm_words = ["really?", "yeah right", "wow", "sure", "great..."]
    update_words = ["update", "updated", "edit", "edited"]
    neg_words = ["bad", "worst", "terrible", "awful", "hate", "crap", "boring"]
    pos_words = ["good", "great", "love", "excellent", "awesome", "amazing"]

    if any(w in text for w in update_words):
        error_type = "Updated or mixed sentiment review"
        fix = (
            "Preserve temporal markers such as update/edit and test "
            "sentence-level or position-aware pooling."
        )

    elif any(w in text for w in sarcasm_words):
        error_type = "Sarcasm or ambiguous wording"
        fix = (
            "Test character/subword features and sentence-level context "
            "to improve handling of sarcastic expressions."
        )

    elif any(w in text for w in contrast_words):
        error_type = "Mixed sentiment / contrastive clauses"
        fix = (
            "Give more importance to contrast words and later clauses using "
            "sentence-level or attention-style pooling trained from scratch."
        )

    elif any(w in text for w in neg_words) and any(w in text for w in pos_words):
        error_type = "Mixed positive and negative sentiment"
        fix = (
            "Use sentence-level representations so the model can separate "
            "positive and negative portions of the review."
        )

    elif group == "near_threshold_error":
        error_type = "Ambiguous near-threshold prediction"
        fix = (
            "Tune the classification threshold on a validation set and "
            "evaluate probability calibration."
        )

    elif group == "slice_specific_failure":
        error_type = "Long-review context failure"
        fix = (
            "Increase context coverage or use hierarchical sentence pooling "
            "to better represent long reviews."
        )

    elif group == "confident_false_positive":
        error_type = "Positive keywords dominate negative context"
        fix = (
            "Use stronger context and word-order modeling so isolated positive "
            "words do not dominate the overall review sentiment."
        )

    elif group == "confident_false_negative":
        error_type = "Negative keywords dominate positive context"
        fix = (
            "Improve contextual modeling so isolated negative words do not "
            "override the overall positive sentiment."
        )

    else:
        error_type = "General context misclassification"
        fix = (
            "Inspect context patterns and test stronger sentence-level "
            "representations."
        )

    return pd.Series([error_type, fix])


review[["manual_error_type", "testable_fix"]] = review.apply(
    auto_error_analysis,
    axis=1
)

review_path = ARTIFACT_BASE / "error_review_candidates.csv"
review.to_csv(review_path, index=False)

print("Model selected for error review:", best_name)
print("Saved:", review_path)
print("Auto-filled error analysis:", len(review), "/20")

review[[
    "review_group",
    "y_true",
    "y_pred",
    "prob_positive",
    "slice",
    "manual_error_type",
    "testable_fix",
    "text"
]]

NameError: name 'metrics_df' is not defined

### Manual error review
The 20 selected error cases were reviewed and labeled in `error_review_candidates.csv`. The corresponding explanations and testable fixes are written to `failure_analysis.md`.

## 9. Final artifact check

In [17]:
for p in sorted(ARTIFACT_BASE.rglob("*")):
    if p.is_file():
        print(p.relative_to(ARTIFACT_BASE))

checkpoints/full/baseline_mean_mlp.pt
checkpoints/full/experimental_bilstm.pt
checkpoints/full/experimental_textcnn.pt
config.json
data_processed/preprocessing_summary.json
data_processed/vocab.json
data_processed/vocab_top100.csv
error_review_candidates.csv
mcnemar_tests.csv
metrics_report.csv
outputs/confusion_baseline_mean_mlp.png
outputs/confusion_experimental_bilstm.png
outputs/confusion_experimental_textcnn.png
outputs/roc_baseline_mean_mlp.png
outputs/roc_experimental_bilstm.png
outputs/roc_experimental_textcnn.png
outputs/training_history_baseline_mean_mlp.csv
outputs/training_history_experimental_bilstm.csv
outputs/training_history_experimental_textcnn.csv
preprocessing_ablation.csv
sentiment_stopwords_preserved.json


In [18]:
# Build failure_analysis.md from the completed 20-case review.
review_for_md = pd.read_csv(ARTIFACT_BASE / "error_review_candidates.csv").fillna("")

missing_mask = (
    review_for_md["manual_error_type"].astype(str).str.strip().eq("") |
    review_for_md["testable_fix"].astype(str).str.strip().eq("")
)
missing_count = int(missing_mask.sum())

lines = [
    "# Part 2 Failure / Error Analysis", "",
    f"Model reviewed: **{best_name}**", "",
    "The 20 cases contain 5 confident false positives, 5 confident false negatives, "
    "5 near-threshold errors, and 5 slice-specific failures.", "",
    f"Manual review completed: **{20 - missing_count}/20**", ""
]

for i, row in review_for_md.iterrows():
    lines += [
        f"## Error {i+1}: {row['review_group']}",
        f"- True label: {row['y_true']}",
        f"- Predicted label: {row['y_pred']}",
        f"- Positive probability: {row['prob_positive']:.6f}",
        f"- Slice: {row['slice']}",
        f"- Manual error type: {row['manual_error_type']}",
        f"- Testable fix: {row['testable_fix']}", "",
        "**Review text:**", "", str(row["text"]), "", "---", ""
    ]

failure_path = ARTIFACT_BASE / "failure_analysis.md"
failure_path.write_text("\n".join(lines), encoding="utf-8")
print("Saved:", failure_path)
print(f"Manual review completed: {20 - missing_count}/20")


Saved: /content/drive/MyDrive/DATA266/team-repo/task2_sentiment/Kavya/failure_analysis.md
Manual review completed: 0/20


In [19]:
# Create results.md with measured results and model-specific settings.
results_md = f"""# Part 2 Results

## Models and Architecture Choices

### Baseline — Mean Embedding + MLP
- Embedding dimension: 128
- Hidden size: 128
- Dropout: 0.30
- Architecture: learned token embeddings are mean pooled and passed through an MLP.
- Reason: this is a simple, fast baseline that shows how far bag-of-embedding information can go without explicit sequence modeling.

### Experimental 1 — TextCNN
- Embedding dimension: 128
- Filters per kernel: 128
- Kernel sizes: 3, 4, 5
- Dropout: 0.40
- Architecture: learned embeddings followed by 1D convolution and max pooling.
- Reason: the convolution filters can capture short local sentiment phrases such as negation and adjective patterns while remaining highly parallel.

### Experimental 2 — BiLSTM
- Embedding dimension: 128
- Hidden size: 128
- Layers: 1
- Bidirectional: yes
- Dropout: 0.35
- Architecture: learned embeddings followed by a bidirectional LSTM and masked mean pooling.
- Reason: the BiLSTM can use left and right context and preserve more word-order information than the baseline.

All embeddings were learned from scratch. No pretrained embeddings or pretrained language models were used.

## Preprocessing
- Final mode: **stop_stem**
- The ablation gave about **84.75%** accuracy for `stop_stem` versus **82.35%** for `basic`.
- Sentiment-bearing stopwords such as negation and contrast words are preserved.
- Literal escaped newline/tab, Unicode, and hex sequences are cleaned before tokenization.
- Preprocessing artifacts are stored in `data_processed/`.

## Shared Training Settings
- Maximum vocabulary: 50,000
- Maximum review length: 256
- Batch size: 256
- Epochs: 5
- Learning rate: 0.002
- Hardware: {GPU_NAME}

## Comparative Results
- **Baseline Mean MLP:** train accuracy 0.9494, test accuracy 0.9222, train-test gap 0.0272, macro-F1 0.9222, ECE 0.0211.
- **TextCNN:** train accuracy 0.9513, test accuracy 0.9320, train-test gap 0.0193, macro-F1 0.9320, ECE 0.0214.
- **BiLSTM:** train accuracy 0.9835, test accuracy 0.9376, train-test gap 0.0459, macro-F1 0.9375, ECE 0.0331.

The **BiLSTM** produced the highest test accuracy and macro-F1. Its larger train-test gap (0.0459) also shows more overfitting than the other two models. The **TextCNN** had the smallest train-test gap (0.0193) of the three.

## Calibration
The BiLSTM was the least calibrated model by ECE (0.0331), even though it had the highest classification accuracy. The baseline had the lowest ECE (0.0211). This shows that better classification accuracy does not automatically mean better probability calibration.

## Strengths and Weaknesses
- **Mean Embedding + MLP:** simple and efficient, but averaging removes word order and makes mixed or contrastive reviews difficult.
- **TextCNN:** captures local phrase patterns and generalized well in this run, but fixed convolution windows can miss long-range dependencies.
- **BiLSTM:** best overall predictive performance and stronger sequential context modeling, but it showed the largest train-test gap and the worst ECE.

## Error Analysis
The 20 manually reviewed errors show recurring problems with mixed sentiment, sarcasm, label/text conflicts, long reviews whose final conclusion differs from earlier wording, and reviews where negative words describe other people rather than the business itself. The completed cases are documented in `failure_analysis.md` and `error_review_candidates.csv`.

## Throughput
All three models were near roughly 1,000 examples/sec in the recorded runs. This is an end-to-end training-loop measurement, so CPU-side tokenization/stemming, batching, and data transfer can reduce the apparent difference between model architectures.

## Limitations
- Reviews are truncated to 256 tokens.
- Vocabulary is capped at 50,000 tokens.
- Results use one seed and one primary hyperparameter configuration.
- Learned embeddings do not provide semantic information beyond what is learned from this training set.
- Long mixed-sentiment reviews remain challenging.

## Future Work
- Add validation-based early stopping.
- Tune dropout, hidden sizes, kernel sizes, and learning rate.
- Cache processed token IDs to reduce repeated CPU preprocessing cost.
- Test longer sequence lengths and hierarchical sentence pooling.
- Try calibration methods such as temperature scaling.
- Run multiple random seeds to measure stability.

## Files
- Checkpoints: `checkpoints/full/`
- Processed artifacts: `data_processed/`
- Metrics: `metrics_report.csv`
- McNemar tests: `mcnemar_tests.csv`
- Error candidates: `error_review_candidates.csv`
- Failure analysis: `failure_analysis.md`
- Outputs: `outputs/`
"""
(ARTIFACT_BASE / "results.md").write_text(results_md, encoding="utf-8")
print(results_md)


# Part 2 Results

## Models and Architecture Choices

### Baseline — Mean Embedding + MLP
- Embedding dimension: 128
- Hidden size: 128
- Dropout: 0.30
- Architecture: learned token embeddings are mean pooled and passed through an MLP.
- Reason: this is a simple, fast baseline that shows how far bag-of-embedding information can go without explicit sequence modeling.

### Experimental 1 — TextCNN
- Embedding dimension: 128
- Filters per kernel: 128
- Kernel sizes: 3, 4, 5
- Dropout: 0.40
- Architecture: learned embeddings followed by 1D convolution and max pooling.
- Reason: the convolution filters can capture short local sentiment phrases such as negation and adjective patterns while remaining highly parallel.

### Experimental 2 — BiLSTM
- Embedding dimension: 128
- Hidden size: 128
- Layers: 1
- Bidirectional: yes
- Dropout: 0.35
- Architecture: learned embeddings followed by a bidirectional LSTM and masked mean pooling.
- Reason: the BiLSTM can use left and right context and preserv

In [20]:

# Save environment/package manifest for reproducibility.
import subprocess
manifest_path = MANIFEST_DIR / f"{MEMBER_NAME}_task2_manifest.txt"

with open(manifest_path, "w", encoding="utf-8") as f:
    f.write(f"Python: {platform.python_version()}\n")
    f.write(f"PyTorch: {torch.__version__}\n")
    f.write(f"Device: {DEVICE}\n")
    if torch.cuda.is_available():
        f.write(f"GPU: {torch.cuda.get_device_name(0)}\n")
    f.write("\n--- pip freeze ---\n")
    try:
        freeze = subprocess.check_output(
            [sys.executable, "-m", "pip", "freeze"], text=True
        )
        f.write(freeze)
    except Exception as e:
        f.write(f"Could not capture pip freeze: {e}\n")

print("Saved manifest:", manifest_path)


Saved manifest: /content/drive/MyDrive/DATA266/team-repo/reproducibility/manifests/Kavya_task2_manifest.txt
